# Three main problems to solve:

    1 MAJOR problem:
        1. How can I group the string in lines?
            If there is a break in the structure it counts as a new line.
            If the power-intensity value changes it counts as a new line.
            If the strings are from a different PST, it counts as a new line.

    2 MINOR problems:
        1. How can I count the number of modules with their power-intensity in each line?
        2. How can I print the text in the middle of the strucutre line?

In [154]:
import numpy as np
import pandas as pd


strings_df = pd.read_excel(r"C:\Users\Usuario\Desktop\Pessoal\Python\Ecotechne\Tagueamento Potência e Intensidade\Sendim estruturas.xlsx")


strings_df['UFV+PST'] = strings_df['UFV'].astype(str)+"."+strings_df['PST'].astype(str)
strings_df['Power'] = strings_df['POT'].astype(str).str[:3] + " Wp"
strings_df['Intensity'] = "F" + strings_df['POT'].astype(str).str[-1]

strings_df['Line'] = np.where(
    strings_df["Type"] == "C",
    round(strings_df["Position Y"]),
    np.where(
        strings_df["Type"] == "T",
        round(strings_df["Position Y"] - 1.1513),
        round(strings_df["Position Y"] + 1.1513)
    )
)


strings_df.sort_values(by=["Line", "Position X"], ascending=[False,True], inplace=True, ignore_index=True)
print(strings_df)

                 Name                                  Layer  UFV  PST  INV  \
0     EMF302_string_1  EMF_eletromec_estrutura_fixa_715Wp_F3    8    2    1   
1     EMF302_string_2  EMF_eletromec_estrutura_fixa_715Wp_F3    8    2    1   
2     EMF302_string_2  EMF_eletromec_estrutura_fixa_715Wp_F3    8    2    1   
3     EMF302_string_2  EMF_eletromec_estrutura_fixa_715Wp_F3    8    2    2   
4     EMF302_string_2  EMF_eletromec_estrutura_fixa_715Wp_F3    8    2    2   
...               ...                                    ...  ...  ...  ...   
5632  EMF302_string_2  EMF_eletromec_estrutura_fixa_705Wp_F2    1    3   19   
5633  EMF302_string_1  EMF_eletromec_estrutura_fixa_705Wp_F2    1    3   19   
5634  EMF302_string_2  EMF_eletromec_estrutura_fixa_705Wp_F2    1    3   19   
5635  EMF302_string_2  EMF_eletromec_estrutura_fixa_705Wp_F2    1    3   19   
5636  EMF302_string_1  EMF_eletromec_estrutura_fixa_705Wp_F2    1    3   19   

      STR  EFX     GRP    GRP+STR    POT Type  Type

In [155]:
'''
MAJOR PROBLEM 1
'''


#Different Lines
condition1 = strings_df["Line"] != strings_df["Line"].shift()

#Break in the Structure with no C type on the edges
condition2 = ((abs(strings_df["Position X"] - strings_df["Position X"].shift()) > 40) &
              (strings_df["Type"] != "C") &
              (strings_df["Type"].shift() != "C"))

#Break in the Structure with one C type on the edges
condition3 = ((abs(strings_df["Position X"] - strings_df["Position X"].shift()) > 30) &
              ((strings_df["Type"] == "C") != (strings_df["Type"].shift() == "C")))

#Break in the Structure with two C types on the edges
condition4 = ((abs(strings_df["Position X"] - strings_df["Position X"].shift()) > 20) &
              (strings_df["Type"] == "C") &
              (strings_df["Type"].shift() == "C"))

#Different PSTs
condition5 = strings_df["UFV+PST"] != strings_df["UFV+PST"].shift()

#Different Power-Intensity
condition6 = strings_df["POT"] != strings_df["POT"].shift()



break_condition = condition1 | condition2 | condition3 | condition4 | condition5 | condition6

strings_df['Group'] = break_condition.cumsum()


In [156]:
'''
MINOR PROBLEM 1
'''

strings_df["Module Count"] = strings_df.groupby(by="Group", sort=False)["POT"].transform('count') * 30
print(strings_df["Module Count"])

0        90
1        90
2        90
3       180
4       180
       ... 
5632     90
5633     90
5634     90
5635     90
5636     90
Name: Module Count, Length: 5637, dtype: int64


In [157]:
'''
MINOR PROBLEM 2
'''


strings_df["Text Position X"] = strings_df.groupby(by="Group", sort=False)["Position X"].transform('mean')
strings_df["Text Position Y"] = strings_df.groupby(by="Group", sort=False)["Position Y"].transform('mean')



strings_df["Script TAG"] = "(command-s \"._mtext\" \""+ strings_df['Text Position X'].astype(str) + "," + \
                     strings_df['Text Position Y'].astype(str) + \
                     "\" \"h\" 1.5  \"j\" \"MC\" \"r\" 0  \"w\" 0 \"" + \
                     strings_df['Power'].astype(str) + " " + strings_df['Intensity'].astype(str) + " - " + strings_df['Module Count'].astype(str) + " modules"\
                     "\" \"\")"

strings_df["Script"] = "(command-s \"._mtext\" \""+ strings_df['Text Position X'].astype(str) + "," + \
                     strings_df['Text Position Y'].astype(str) + \
                     "\" \"h\" 1.5  \"j\" \"MC\" \"r\" 0  \"w\" 0 \"" + \
                     strings_df['Group'].astype(str) +\
                     "\" \"\")"


strings_df_unique = list(set(strings_df["Script TAG"]))

with open (r"C:\Users\Usuario\Desktop\Pessoal\Python\Ecotechne\Tagueamento Potência e Intensidade\Sendim - Tagueamento Potência e Intensidade.scr", 'w') as f:
    f.write('\n'.join(strings_df_unique) + '\n')

with open (r"C:\Users\Usuario\Desktop\Pessoal\Python\Ecotechne\Tagueamento Potência e Intensidade\teste.scr", 'w') as f:
    f.write('\n'.join(strings_df["Script"].astype(str)) + '\n')



strings_df.to_excel(r"C:\Users\Usuario\Desktop\Pessoal\Python\Ecotechne\Tagueamento Potência e Intensidade\Sendim - Tagueamento Potência e Intensidade.xlsx")
print(strings_df)

                 Name                                  Layer  UFV  PST  INV  \
0     EMF302_string_1  EMF_eletromec_estrutura_fixa_715Wp_F3    8    2    1   
1     EMF302_string_2  EMF_eletromec_estrutura_fixa_715Wp_F3    8    2    1   
2     EMF302_string_2  EMF_eletromec_estrutura_fixa_715Wp_F3    8    2    1   
3     EMF302_string_2  EMF_eletromec_estrutura_fixa_715Wp_F3    8    2    2   
4     EMF302_string_2  EMF_eletromec_estrutura_fixa_715Wp_F3    8    2    2   
...               ...                                    ...  ...  ...  ...   
5632  EMF302_string_2  EMF_eletromec_estrutura_fixa_705Wp_F2    1    3   19   
5633  EMF302_string_1  EMF_eletromec_estrutura_fixa_705Wp_F2    1    3   19   
5634  EMF302_string_2  EMF_eletromec_estrutura_fixa_705Wp_F2    1    3   19   
5635  EMF302_string_2  EMF_eletromec_estrutura_fixa_705Wp_F2    1    3   19   
5636  EMF302_string_1  EMF_eletromec_estrutura_fixa_705Wp_F2    1    3   19   

      STR  EFX     GRP    GRP+STR    POT  ... UFV+P